In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from model_tuner import loadObjects
from model_metrics import (
    combine_plots,
    show_roc_curve,
    show_pr_curve,
    show_confusion_matrix,
    show_calibration_curve,
    summarize_model_performance
)


## Paths

In [ ]:
data_path = "../data/processed"

In [ ]:
from eda_toolkit import ensure_directory
import os  # import operating system for dir

base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_pdf = os.path.join(base_path, "images", "pdf_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_pdf)
ensure_directory(image_path_svg)

In [ ]:
import json
from pathlib import Path

pred_dir = Path("../models/predictions/full_text_clean")

X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

In [ ]:
from model_metrics.model_registry import set_stores

set_stores("mlruns/models")                     # only the live store

In [ ]:
from model_metrics.model_registry import best_per_algo, load_best_per_algo

best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")

In [ ]:
champs

In [ ]:
model_catboost = champs["cat_outcome"]
model_catboost_no_sex = champs["cat_outcome_no_sex"]
model_xgboost = champs["xgb_outcome"]
model_rf = champs["rf_outcome"]
model_lr = champs["lr_outcome"]

In [ ]:
model_titles = ["CatBoost", "CatBoost (sex removed)", "XGBoost", "Random Forest", "Logistic Regression"]
models = [model_catboost, model_catboost_no_sex, model_xgboost, model_rf, model_lr,]

In [ ]:
thresholds = {
    "Logistic Regression": next(iter(model_lr.threshold.values())),
    "Random Forest": next(iter(model_rf.threshold.values())),
    "XGBoost": next(iter(model_xgboost.threshold.values())),
    "CatBoost": next(iter(model_catboost.threshold.values())),
    "CatBoost (sex removed)": next(iter(model_catboost_no_sex.threshold.values())),
}

In [ ]:
X_valid, y_valid = model_lr.get_valid_data(X, y)
X_test, y_test = model_lr.get_test_data(X, y)
y_test = y_test["outcome"]

In [ ]:
from model_metrics import get_expected_features

get_expected_features(model_rf)

In [ ]:
from model_metrics import summarize_model_performance

model_performance = summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    model_type="classification",
    return_df=True,
    model_threshold=thresholds,
)

model_performance

In [ ]:
from model_tuner import evaluate_bootstrap_metrics

boot_metrics = [
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1",
    "neg_brier_score",
]

y_bs = pd.Series(np.asarray(y_test).ravel()).reset_index(drop=True)

results = []
for model, title in zip(models, model_titles):
    y_prob = pd.Series(model.predict_proba(X_test)[:, 1]).reset_index(drop=True)

    df = evaluate_bootstrap_metrics(
        y=y_bs,
        y_pred_prob=y_prob,
        metrics=boot_metrics,
        threshold=thresholds[title],
        n_samples=len(y_bs),
        num_resamples=5000,
        stratify=y_bs,
        model_type="classification",
        ci_method="percentile",
        random_state=222,
    )
    df.insert(0, "Model", title)
    results.append(df)

bootstrap_performance = pd.concat(results, ignore_index=True)

Table 2: point estimates with percentile bootstrap confidence intervals.

`evaluate_bootstrap_metrics` returns the bootstrap MEAN as its central value.
The reported centre is the observed statistic on the test set instead, for
two reasons:

  1. It is the quantity a reader wants. The bootstrap mean is a property of
     the resampling procedure, not of the model.
  2. Bootstrap means of average precision are biased upward on small samples.
     Here that inflates every model, for example CatBoost 0.304 -> 0.321, so
     reporting the observed value is the more conservative choice.

Pairing an observed statistic with a percentile bootstrap interval is the
conventional form in clinical prediction reporting. The bootstrap mean is
retained in `bootstrap_performance` for reference but is not what Table 2
reports.

Group differences in Tables 3 and 4 DO report bootstrap means, because a
difference relative to a reference group has no direct point-estimate
equivalent. Methods 2.6 states both conventions.

In [ ]:
bootstrap_performance.round(3)

In [ ]:
from model_metrics import show_roc_curve

show_roc_curve(
    model=models,
    X=X_test,
    y=y_test,
    model_title=model_titles,
    decimal_places=2,
    curve_kwgs={
        "CatBoost": {"color": "green", "linewidth": 1},
        "CatBoost (sex removed)": {"color": "orange", "linewidth": 1},
        "XGBoost": {"color": "purple", "linewidth": 1},
        "Random Forest": {"color": "black", "linewidth": 1},
        "Logistic Regression": {"color": "blue", "linewidth": 1},
    },
    linestyle_kwgs={"color": "red", "linestyle": "--"},
    title="ROC Curves: Logistic Regression and Random Forest",
    overlay=True,
    image_filename=os.path.join(image_path_png, "roc_curves.pdf")
)

In [ ]:
curve_kwgs = {
    "CatBoost": {"color": "blue", "linewidth": 1.5},
    "CatBoost (sex removed)": {"color": "orange", "linewidth": 1.5},
    "XGBoost": {"color": "purple", "linewidth": 1.5},
    "Random Forest": {"color": "black", "linewidth": 1.5},
    "Logistic Regression": {"color": "green", "linewidth": 1.5},
}

combine_plots(
    plot_calls=[
        (
            show_roc_curve,
            {
                "model": models,
                "X": X_test,
                "y": y_test,
                "model_title": model_titles,
                "overlay": True,
                "curve_kwgs": curve_kwgs,
                "title": "ROC Curves: All Models",
            },
        ),
        (
            show_pr_curve,
            {
                "model": models,
                "X": X_test,
                "y": y_test,
                "model_title": model_titles,
                "legend_loc": "right",
                "overlay": True,
                "curve_kwgs": curve_kwgs,
                "title": "Precision-Recall Curves: All Models",
            },
        ),
    ],
    image_filename="../images/pdf_images/roc_pr_curves.pdf",
)

In [ ]:
import math

n_cols = 2
n_rows = math.ceil(len(models) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()

for ax, m, t in zip(axes, models, model_titles):
    show_confusion_matrix(
        model=m,
        X=X_test,
        y=y_test,
        model_title=t,
        title=f"{t} (Threshold = {thresholds[t]:.2f})",
        model_threshold=thresholds[t],
        text_wrap=50,
        ax=ax,
    )

for ax in axes[len(models):]:
    ax.axis("off")

plt.tight_layout()

# save the assembled figure once, not once per panel
fig.savefig(os.path.join(image_path_pdf, "show_confusion_matrix.pdf"),
            bbox_inches="tight")
fig.savefig(os.path.join(image_path_png, "show_confusion_matrix.png"),
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

fig, ax = plt.subplots(figsize=(8, 4))

plot_threshold_metrics(
    model=model_catboost,
    X_test=X_test,
    y_test=y_test["outcome"] if hasattr(y_test, "columns") else y_test,
    baseline_thresh=False,
    baseline_kwgs={"color": "purple", "linestyle": "--", "linewidth": 1.5},
    curve_kwgs={"linestyle": "-", "linewidth": 1.5},
    title="CatBoost threshold metrics: precision, sensitivity, specificity, F1",
    text_wrap=40,
    model_threshold=next(iter(model_catboost.threshold.values())),
    ax=ax,
)

# the library labels the curve "Recall"; the manuscript uses "Sensitivity"
leg = ax.get_legend()
if leg is not None:
    for txt in leg.get_texts():
        if txt.get_text() == "Recall":
            txt.set_text("Sensitivity")

plt.tight_layout()

for path in (image_path_pdf, image_path_png):
    Path(path).mkdir(parents=True, exist_ok=True)

fig.savefig(os.path.join(image_path_pdf, "threshold_metrics.pdf"),
            bbox_inches="tight")
fig.savefig(os.path.join(image_path_png, "threshold_metrics.png"),
            dpi=300, bbox_inches="tight")
plt.show()